[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%3A%20NLP/01_Texto_y_maquina.ipynb)


# ¿Cómo capta el lenguaje una máquina?

Antes de hablar de modelos, algoritmos o inteligencia artificial, hay una pregunta más fundamental:

> **¿Qué ve exactamente una computadora cuando lee un texto?**

La respuesta corta: **nada**. Una computadora solo entiende números. Todo lo demás (letras, palabras, frases, emojis) tiene que traducirse a números antes de que la máquina pueda hacer algo con ello.

Ese problema de traducción es, en esencia, el punto de partida del **Procesamiento de Lenguaje Natural (NLP)**.

En este notebook vamos a recorrer las distintas formas en que se ha intentado resolver ese problema. Cada solución funciona... hasta que deja de funcionar. Y ese momento es exactamente donde empieza la siguiente idea.

---

## El texto con el que vamos a trabajar

Vamos a usar las mismas dos líneas a lo largo de todo el notebook. No es un texto al azar: fue elegido porque contiene casi todos los problemas que vamos a encontrar.


In [1]:
linea_1 = "El 80% de los datos del mundo no están estructurados: imágenes, audios y sobre todo texto. 📊"
linea_2 = "¿Puede una máquina 'leer' eso? Ése es, precisamente, el reto del Procesamiento de Lenguaje Natural. 🤖"

texto = linea_1 + "\n" + linea_2
print(texto)

El 80% de los datos del mundo no están estructurados: imágenes, audios y sobre todo texto. 📊
¿Puede una máquina 'leer' eso? Ése es, precisamente, el reto del Procesamiento de Lenguaje Natural. 🤖


A simple vista parece sencillo. Ahora veamos lo que ve la máquina.

---

## Nivel 1. La unidad más pequeña: el caracter

La forma más directa de convertir texto en números es asignarle un número a cada caracter. Esto no es una invención reciente: en los años 60 se creó el estándar **ASCII** (*American Standard Code for Information Interchange*), que mapeó 128 caracteres (letras del inglés, números y signos de puntuación básicos) a números del 0 al 127.

En Python podemos ver ese número con la función `ord()`:

In [11]:
print(ord("🤖"))

129302


In [8]:
# Cada caracter tiene un número asignado
for caracter in "Hola Ibero":
    print(f"  '{caracter}'  →  {ord(caracter)}")

  'H'  →  72
  'o'  →  111
  'l'  →  108
  'a'  →  97
  ' '  →  32
  'I'  →  73
  'b'  →  98
  'e'  →  101
  'r'  →  114
  'o'  →  111


### El problema inmediato: ASCII solo conoce el inglés

ASCII fue diseñado para el inglés. Intentemos procesar nuestro texto con esa lógica:

In [10]:
# ¿Qué pasa si intentamos codificar nuestro texto solo con ASCII?
try:
    print(linea_1)
    linea_1.encode('ascii')
except UnicodeEncodeError as e:
    print(f"Error: {e}")
    print("\nASCII no puede representar caracteres como á, é, ñ, ¿ o emojis.")

El 80% de los datos del mundo no están estructurados: imágenes, audios y sobre todo texto. 📊
Error: 'ascii' codec can't encode character '\xe1' in position 36: ordinal not in range(128)

ASCII no puede representar caracteres como á, é, ñ, ¿ o emojis.


### La solución: Unicode y UTF-8

**Unicode** es un estándar moderno que asigna un número único a más de 140,000 caracteres de todos los idiomas del mundo, incluyendo emojis. **UTF-8** es la forma más común de *codificar* esos números en bytes.

Con Unicode, nuestro texto ya no falla. Pero aparece algo nuevo:

In [12]:
# Comparemos cuánto espacio ocupa cada tipo de caracter en UTF-8
caracteres_de_prueba = ['a', 'á', 'ñ', '¿', '—', '📊', '🤖']

print(f"{'Caracter':<12} {'Número Unicode':<18} {'Bytes en UTF-8':<18}")
print("-" * 50)
for c in caracteres_de_prueba:
    encoded = c.encode('utf-8')
    print(f"  '{c}'  {'':<8} U+{ord(c):04X}  {'':<10} {len(encoded)} byte(s)  {'':<8}")

Caracter     Número Unicode     Bytes en UTF-8    
--------------------------------------------------
  'a'           U+0061             1 byte(s)          
  'á'           U+00E1             2 byte(s)          
  'ñ'           U+00F1             2 byte(s)          
  '¿'           U+00BF             2 byte(s)          
  '—'           U+2014             3 byte(s)          
  '📊'           U+1F4CA             4 byte(s)          
  '🤖'           U+1F916             4 byte(s)          


### Los problemas que quedan abiertos

Aunque Unicode resuelve la representación de caracteres, trabajar a nivel de **caracter individual** tiene costos altos:

| Problema | Ejemplo |
|---|---|
| Mayúsculas y minúsculas son números distintos | `'E'` = 69, `'e'` = 101 |
| Las vocales acentuadas son un número diferente | `'a'` = 97, `'á'` = 225 |
| Un emoji puede ocupar 4 bytes (4 "caracteres" para la máquina) | `'📊'` → 4 bytes |
| El guión largo (U+2014) no es el mismo que el guión corto `-` | U+2014 vs U+002D |
| **Ningún número captura significado** | `'g'`, `'a'`, `'t'`, `'o'` son cuatro números sin relación entre sí |

Ese último punto es el más importante: aunque representemos perfectamente cada caracter, la máquina no sabe que `g-a-t-o` es un animal.

**¿Y si en lugar de caracteres usamos palabras completas?**

---

## Nivel 2. Palabras como unidad: el vocabulario

La siguiente idea es más intuitiva: en lugar de asignar un número a cada letra, asignamos un número a cada **palabra**. Construimos un diccionario (un vocabulario) y cada palabra recibe un índice único.

Empecemos con lo más simple: separar el texto por espacios.

In [13]:
print(linea_1)

El 80% de los datos del mundo no están estructurados: imágenes, audios y sobre todo texto. 📊


In [14]:
# Separar por espacios — la forma más ingenua
tokens_crudos = texto.split()

print(f"Total de tokens: {len(tokens_crudos)}")
print("\nLo que obtenemos:")
for i, t in enumerate(tokens_crudos):
    print(f"  [{i}] '{t}'")

Total de tokens: 33

Lo que obtenemos:
  [0] 'El'
  [1] '80%'
  [2] 'de'
  [3] 'los'
  [4] 'datos'
  [5] 'del'
  [6] 'mundo'
  [7] 'no'
  [8] 'están'
  [9] 'estructurados:'
  [10] 'imágenes,'
  [11] 'audios'
  [12] 'y'
  [13] 'sobre'
  [14] 'todo'
  [15] 'texto.'
  [16] '📊'
  [17] '¿Puede'
  [18] 'una'
  [19] 'máquina'
  [20] ''leer''
  [21] 'eso?'
  [22] 'Ése'
  [23] 'es,'
  [24] 'precisamente,'
  [25] 'el'
  [26] 'reto'
  [27] 'del'
  [28] 'Procesamiento'
  [29] 'de'
  [30] 'Lenguaje'
  [31] 'Natural.'
  [32] '🤖'


Antes de construir el vocabulario, hay que limpiar. De lo contrario, `"texto."` y `"texto"` serían dos palabras distintas para la máquina.

Para limpiar texto de forma flexible usamos **expresiones regulares** (regex). Una expresión regular es un patrón que describe qué caracteres queremos encontrar o eliminar.

La función principal es `re.sub(patrón, reemplazo, texto)`: busca el patrón en el texto y lo sustituye. Veamos los bloques básicos:

| Patrón | Qué coincide | Ejemplo |
|---|---|---|
| `\d` | Cualquier dígito 0-9 | `\d+` encuentra `"80"` en `"el 80%"` |
| `\s` | Cualquier espacio en blanco | `\s+` colapsa espacios múltiples |
| `[abc]` | Cualquiera de los caracteres listados | `[aeiou]` encuentra cualquier vocal |
| `[.,;:!?]` | Cualquiera de esos signos de puntuación | útil para limpiar texto |


In [15]:
import re

# Ejemplo 1: encontrar patrones con re.findall
frase = "El 80% de los datos y los 3 formatos principales"


numeros = re.findall(r"\d+", frase)
print("Dígitos encontrados:", numeros)

Dígitos encontrados: ['80', '3']


In [16]:
# Ejemplo 2: eliminar signos de puntuación específicos
frase2 = "Hola, ¿cómo estás? ¡Muy bien!"

sin_signos = re.sub(r"[¿¡!?,.]", "", frase2)
print(sin_signos)

Hola cómo estás Muy bien


In [17]:
# Ejemplo 3: colapsar espacios múltiples
frase3 = "El   texto   tiene    espacios   extra"

frase_limpia = re.sub(r"\s+", " ", frase3).strip()
print(frase_limpia)

El texto tiene espacios extra


In [18]:
# Ejemplo 4: eliminar cualquier caracter que NO sea letra, número o espacio
frase4 = "precio: $150 - ¡oferta especial!"

solo_palabras = re.sub(r"[^a-záéíóúüñA-ZÁÉÍÓÚÜÑ0-9 ]", "", frase4)
print(solo_palabras)

precio 150  oferta especial


In [ ]:
import re

# Limpieza básica: minúsculas y sin puntuación simple
# (por ahora dejamos los emojis para ver qué pasa)
texto_limpio = texto.lower()
texto_limpio = re.sub(r"[.,;:!?¿¡'\"()\-—]", "", texto_limpio)

palabras = texto_limpio.split()

print(f"Palabras después de limpiar: {len(palabras)}")
print(palabras)

In [ ]:
# Construir el vocabulario: cada palabra única recibe un número
vocabulario = {palabra: indice for indice, palabra in enumerate(sorted(set(palabras)))}

print(f"Tamaño del vocabulario: {len(vocabulario)} palabras únicas")
print("\nVocabulario:")
for palabra, indice in vocabulario.items():
    print(f"  {indice:>3}  →  '{palabra}'")

In [ ]:
# Así "ve" la máquina el texto como secuencia de índices
texto_como_numeros = [vocabulario[p] for p in palabras]
print("El texto como secuencia de índices:")
print(texto_como_numeros)

### El problema del vocabulario: escala

Con nuestras dos líneas el vocabulario es manejable. Pero pensemos en escala real:

- La RAE registra ~**88,000** palabras en español
- Con conjugaciones, plurales y variantes regionales: fácilmente **millones** de formas
- `"leer"`, `"leyendo"`, `"leyó"`, `"leída"` son **cuatro entradas distintas** en el vocabulario, aunque describen lo mismo
- Una palabra nueva, un nombre propio, un término técnico o un error tipográfico → **no existe** en el vocabulario
- Los emojis `📊` y `🤖` tampoco tienen entrada

Este último punto tiene nombre: se llama el problema de las palabras **fuera de vocabulario** (*Out-Of-Vocabulary*, OOV).

Veámoslo:

In [ ]:
# ¿Qué pasa con una palabra que no está en el vocabulario?
nueva_oracion = "los podcasts y las newsletters también son datos no estructurados"

palabras_nuevas = nueva_oracion.split()
oov = [p for p in palabras_nuevas if p not in vocabulario]

print(f"Palabras en la nueva oración: {palabras_nuevas}")
print(f"\nPalabras que NO están en nuestro vocabulario (OOV): {oov}")
print("\nLa máquina no sabe qué hacer con ellas.")

### Los problemas que quedan abiertos

| Problema | Por qué importa |
|---|---|
| Vocabulario enorme | Más memoria, más cómputo, más difícil de entrenar |
| Palabras OOV | Una sola palabra desconocida puede romper el sistema |
| Morfología del español | `"estructurado"` y `"estructurados"` son entradas distintas |
| Orden sin significado | Los índices `[4, 12, 7]` no dicen nada sobre la relación entre palabras |

**¿Y si la unidad no fuera ni el caracter ni la palabra completa, sino algo intermedio?**

---

## Nivel 3. Subwords: el término medio

La idea es partir las palabras en **fragmentos reutilizables**. No tan pequeño como un caracter (sin significado), no tan grande como una palabra completa (vocabulario inmanejable).

Por ejemplo:

```
"estructurados"  →  "estruc" + "tura" + "dos"
"estructurada"   →  "estruc" + "tura" + "da"
"reestructurar"  →  "re" + "estruc" + "tura" + "r"
```

El prefijo `"estruc"` y la raíz `"tura"` aparecen en las tres. Con un vocabulario de fragmentos pequeño, podemos representar casi cualquier palabra, incluso las que nunca hemos visto.

Este principio está detrás de algoritmos como **BPE** (*Byte Pair Encoding*), que es el mismo que usan modelos como GPT o BERT para tokenizar texto. Lo exploraremos en profundidad en un notebook posterior. Por ahora, veamos la intuición:

In [ ]:
# Intuición manual de subpalabras
# (esto NO es BPE real, es solo para ilustrar la idea)

fragmentos_conocidos = ["estruc", "tura", "do", "dos", "da", "das", "re", "des", "r", "s"]

def tokenizar_subpalabra_manual(palabra, fragmentos):
    """Intenta cubrir la palabra con fragmentos conocidos"""
    resultado = []
    i = 0
    while i < len(palabra):
        encontrado = False
        for f in sorted(fragmentos, key=len, reverse=True):  # Fragmentos más largos primero
            if palabra[i:].startswith(f):
                resultado.append(f)
                i += len(f)
                encontrado = True
                break
        if not encontrado:
            resultado.append(palabra[i])  # caracter individual como fallback
            i += 1
    return resultado

palabras_test = ["estructurados", "estructurada", "reestructurar", "desestructuradas", "podcast"]

for p in palabras_test:
    tokens = tokenizar_subpalabra_manual(p, fragmentos_conocidos)
    print(f"  '{p}'  →  {tokens}")

Observa que `"podcast"`, que no estaba en nuestro vocabulario, se puede representar igual, aunque sea con caracteres individuales como fallback. El sistema nunca queda paralizado.

### Los problemas que quedan abiertos

Las subpalabras resuelven el problema OOV y reducen el vocabulario, pero:

- Los fragmentos ya no son intuitivos: `"estruc"` no significa nada por sí solo
- Elegir cómo partir las palabras requiere aprender del corpus (no es trivial)
- **Aún no hay significado**: la máquina tiene mejores índices, pero sigue sin saber que `"texto"` y `"documento"` son cosas parecidas

Antes de atacar el significado, hay algo más que podemos hacer con las representaciones que ya tenemos: **limpiar**.

---

## Nivel 4. Stop Words: no todas las palabras valen lo mismo

Si analizamos cualquier texto en español, hay palabras que aparecen constantemente: `"el"`, `"la"`, `"de"`, `"que"`, `"es"`, `"y"`. Son tan frecuentes que no aportan información específica sobre el tema del texto.

Se les llama **stop words** (palabras vacías). Eliminarlas reduce el ruido.

Veamos qué queda de nuestro texto si las quitamos:

In [ ]:
stop_words = {
    "el", "la", "los", "las", "de", "del", "que", "es", "en", "y",
    "no", "eso", "una", "un", "se", "su", "por", "con", "para",
    "lo", "le", "al", "a", "e"
}

palabras_sin_stop = [p for p in palabras if p not in stop_words]

print("Texto original (palabras):")
print(" | ".join(palabras))

print(f"\nDespués de eliminar stop words ({len(palabras)} → {len(palabras_sin_stop)} tokens):")
print(" | ".join(palabras_sin_stop))

print("\n¿Se conserva el significado principal?")

### El problema: el contexto importa

Eliminar stop words funciona bien para tareas como **búsqueda** o **clasificación de temas**. Pero hay contextos donde destruye información crítica:

| Oración original | Sin stop words | Problema |
|---|---|---|
| `"el vino tinto"` | `"vino tinto"` | ¿El sustantivo o el verbo? |
| `"no me gustó"` | `"gustó"` | ¡Se perdió la negación! |
| `"no es lo que esperaba"` | `"esperaba"` | Sentido completamente opuesto |

La decisión de eliminar stop words **depende de la tarea**. No es una regla universal.

---

## Nivel 5. Normalización: Stemming y Lematización

Tenemos otro problema: `"estructurado"`, `"estructurados"`, `"estructurada"` y `"desestructurar"` son cuatro entradas distintas en el vocabulario, aunque comparten la misma raíz y un significado relacionado.

Hay dos formas de normalizarlas.

### Stemming: cortar hasta la raíz

El **stemming** aplica reglas mecánicas para recortar las palabras hasta su supuesta raíz. Es rápido, pero no siempre produce una palabra real.

In [ ]:
# Stemming manual simplificado — solo para ilustrar la idea
# (en producción se usa SnowballStemmer de NLTK)

sufijos_a_eliminar = ["ados", "adas", "ado", "ada", "ando", "iendo",
                       "ción", "ciones", "mente", "es", "s"]

def stem_simple(palabra):
    for sufijo in sufijos_a_eliminar:
        if palabra.endswith(sufijo) and len(palabra) > len(sufijo) + 3:
            return palabra[:-len(sufijo)]
    return palabra

ejemplos = ["estructurados", "estructurada", "procesamiento",
            "precisamente", "imágenes", "audios", "natural"]

print(f"{'Palabra original':<22} → {'Stem'}")
print("-" * 38)
for p in ejemplos:
    print(f"  {p:<20} → {stem_simple(p)}")

### Lematización: la forma base real

La **lematización** es más sofisticada: en lugar de cortar mecánicamente, busca la forma canónica de la palabra usando conocimiento lingüístico.

| Palabra | Stem (tosco) | Lema (correcto) |
|---|---|---|
| `"durmiendo"` | `"durmien"` | `"dormir"` |
| `"fui"` | `"fu"` | `"ir"` |
| `"imágenes"` | `"imágen"` | `"imagen"` |
| `"precisamente"` | `"precis"` | `"preciso"` |

La lematización requiere un diccionario y, en muchos casos, saber el **contexto** de la palabra. Librerías como `spaCy` la implementan para español.

### El problema que queda abierto

Tanto el stemming como la lematización reducen el vocabulario y unifican formas distintas de la misma idea. Pero siguen sin resolver algo esencial:

- `"texto"` y `"documento"` tienen lemas distintos, pero **significan cosas muy parecidas**
- `"Natural"` en *"Lenguaje Natural"* significa algo completamente distinto que en *"parque natural"*
- `"banco"` puede ser una institución financiera o un mueble — el lema es el mismo

El **significado** depende del **contexto**, y ninguna de las técnicas que hemos visto lo captura.

---

## Resumen: el recorrido hasta aquí

| Técnica | Qué resuelve | Qué deja abierto |
|---|---|---|
| **Caracteres (Unicode)** | Representar cualquier símbolo como número | Sin significado; mayúsculas, acentos y emojis complican todo |
| **Vocabulario de palabras** | Unidades con significado léxico | Vocabulario enorme; palabras OOV; morfología del español |
| **Subpalabras** | Vocabulario manejable; sin OOV | Fragmentos no intuitivos; aún sin significado |
| **Stop words** | Reduce ruido; mejora tareas de búsqueda | Destruye negaciones y contexto en otras tareas |
| **Stemming / Lematización** | Unifica formas distintas de la misma idea | No captura similitud semántica ni ambigüedad por contexto |

---

## La pregunta que queda abierta

Llegamos hasta aquí sabiendo **cómo representar** texto como números. Eso ya es mucho.

Pero representar no es comprender. La máquina puede tener los índices `[4, 12, 7, 23]` perfectamente ordenados y aun así no saber que:

- `"texto"` y `"documento"` son casi lo mismo
- `"no me gustó"` es lo opuesto de `"me gustó"`
- `"banco"` significa cosas distintas según lo que le rodea

Para capturar eso necesitamos otra cosa: **vectores que lleven significado**. A eso se le llama *embeddings*, y será el tema de un notebook próximo.

Por ahora, la pregunta con la que cerramos:

> *¿Es posible representar el significado de una palabra como un punto en un espacio matemático, de forma que palabras parecidas queden cerca entre sí?*
>
> La respuesta es sí. Y cambia todo.